# Increasingly Complex Dashboards &mdash; a `nb.dashboard()` tour

This notebook walks up a ladder of dashboards, from the two-line minimum to a full
multi-panel operations board. Each rung adds **one** new idea, so by the end every
feature has been introduced in isolation before it appears in the big board.

| Level | Adds |
|---|---|
| 1 | The minimum &mdash; two panels, shared header |
| 2 | Mixing plot types (line / bar / box / histogram) |
| 3 | Per-panel `kwargs` (stacked bars, normalized histograms) + **pinned** panels |
| 4 | Multi-axis (`plot_ymult`) and a live interpolation **table** |
| 5 | **Contour** engineering maps with overlaid datasets |
| 6 | The everything board &mdash; 3&times;3 grid, dark theme, every panel type |

Every panel here was verified to render real data before it went in the notebook.

## 0. Setup &mdash; three rig runs + a compressor map

Two data families feed the tour:

- **`uc`** &mdash; three runs of a rig logged over a `time` sweep, each with continuous
  channels (`pressure`, `temp`, `flow`, `rpm`, `vibration`) and a categorical `phase`.
  The mix lets one board show lines, bars, boxes, histograms and tables of the *same*
  data.
- **`uc2`** &mdash; a scattered efficiency field plus a surge-line and operating-point
  overlay, for the contour board. Contour data is a different *shape* (scattered
  `x,y,z`), so it lives on its own notebook &mdash; the header's dataset picker assumes
  every panel shares one schema.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

rng = np.random.default_rng(0)

def run(scale):
    t = np.arange(0, 60)
    phase = np.where(t < 15, 'idle', np.where(t < 40, 'ramp', 'peak'))
    return pd.DataFrame({
        'time':      t,
        'pressure':  scale * (10 + 0.8 * t + rng.normal(0, 1.5, t.size)),
        'temp':      scale * (300 + 6 * t + rng.normal(0, 8, t.size)),
        'flow':      scale * (5 + 0.3 * t + rng.normal(0, 0.5, t.size)),
        'rpm':       scale * (1500 + 40 * t + rng.normal(0, 60, t.size)),
        'vibration': scale * (0.5 + 0.02 * t + rng.normal(0, 0.15, t.size)),
        'phase':     phase,
    })

uc = UnichartNotebook()
uc.load(run(1.00), title='Run A')   # Set 0
uc.load(run(1.15), title='Run B')   # Set 1
uc.load(run(0.90), title='Run C')   # Set 2
uc.sets[0].df.head()

UniChart Notebook Environment Initialized.
Loaded Set 0: Run A
Loaded Set 1: Run B
Loaded Set 2: Run C


,time,pressure,temp,flow,rpm,vibration,phase
0,0,10.188595,296.508518,5.393794,1487.728651,0.694537,idle
1,1,10.601843,296.641585,5.722039,1477.428392,0.468149,idle
2,2,12.560634,325.914943,5.637797,1616.787388,0.668188,idle
3,3,12.557350,314.032714,5.186613,1607.980218,0.486655,idle
4,4,12.396496,326.631757,6.132477,1633.787900,0.844100,idle


In [2]:
# --- the compressor-map world for the contour board (Level 5) ---
flow = rng.uniform(2.0, 8.0, 220)
pr   = rng.uniform(1.5, 4.0, 220)
field = pd.DataFrame({
    'FLOW': flow, 'PR': pr,
    'EFF':    0.88 - 0.012 * (flow - 5.0) ** 2 - 0.06 * (pr - 2.6) ** 2,
    'MARGIN': 4.0 - pr,
})
sx = np.linspace(2.2, 7.5, 12)
surge = pd.DataFrame({'FLOW': sx, 'PR': 1.6 + 0.42 * sx})
ops   = pd.DataFrame({'FLOW': [3.5, 4.5, 5.0, 5.5, 6.5],
                      'PR':   [2.0, 2.4, 2.7, 3.0, 3.4]})

uc2 = UnichartNotebook()
uc2.load(field, title='Efficiency field')   # Set 0  (the surface)
uc2.load(surge, title='Surge line')         # Set 1  (line overlay)
uc2.load(ops,   title='Operating points')   # Set 2  (marker overlay)
uc2.color(1, 'black'); uc2.linestyle(1, '-')                    # surge -> solid line
uc2.color(2, 'red');   uc2.marker(2, 'o'); uc2.markersize(2, 13)  # ops -> red circles

UniChart Notebook Environment Initialized.
Loaded Set 0: Efficiency field
Loaded Set 1: Surge line
Loaded Set 2: Operating points


## 1. The minimum

Pass the notebook and a list of **panel specs**. Each spec seeds that panel's initial
controls; `method` defaults to `'plot'`. The call launches a Dash server and renders
inline.

> Try it: untick a **dataset** in the header &mdash; both panels drop it together. That
> shared header is the whole idea: one consistent slice of the data across every panel.

In [3]:
uc.dashboard([
    {'method': 'plot', 'x': 'time', 'y': 'pressure'},
    {'method': 'plot', 'x': 'time', 'y': 'temp'},
])

## 2. Mixing plot types

The same board can carry different **methods**. Continuous channels want lines; a
categorical `phase` wants bars and boxes; one channel's distribution wants a histogram.
`suptitle` names each card (it's drawn in the card header, not inside the figure).

Each panel also keeps a live **plot-type switch**, so any of these can be changed
in-place without touching the code.

In [4]:
uc.dashboard(
    panels=[
        {'method': 'plot',      'x': 'time',      'y': ['pressure', 'temp'], 'suptitle': 'Channels vs time'},
        {'method': 'bar',       'x': 'phase',     'y': 'flow',               'suptitle': 'Mean flow by phase'},
        {'method': 'box',       'x': 'phase',     'y': 'temp',               'suptitle': 'Temp spread by phase'},
        {'method': 'histogram', 'x': 'vibration', 'y': 'vibration',          'suptitle': 'Vibration'},
    ],
    title='Rig overview',
    ncols=2,
    height=360,
)

## 3. Per-panel `kwargs` and pinned panels

Two new controls:

- **`kwargs`** passes method-specific options straight through &mdash; `barmode='stack'`
  and `agg='sum'` for a totals bar, `nbins`/`histnorm='percent'` for a normalized
  histogram, `points='all'` to show every box point. Each kwarg is applied only when
  the active method accepts it, so it *survives* a plot-type switch instead of raising.
- **`datasets`** pins a panel to specific sets. A pinned panel ignores the header
  picker and wears a "pinned" badge &mdash; here the last panel always shows only Run A,
  as a fixed reference while the rest of the board follows the header.

In [5]:
uc.dashboard(
    panels=[
        {'method': 'bar',       'x': 'phase', 'y': ['flow', 'pressure'],
         'kwargs': {'barmode': 'stack', 'agg': 'sum'}, 'suptitle': 'Stacked totals by phase'},
        {'method': 'histogram', 'x': 'temp',  'y': 'temp',
         'kwargs': {'nbins': 30, 'histnorm': 'percent'}, 'suptitle': 'Temp distribution (%)'},
        {'method': 'box',       'x': 'phase', 'y': 'rpm',
         'kwargs': {'points': 'all'}, 'suptitle': 'RPM by phase (all points)'},
        {'method': 'plot',      'x': 'time',  'y': 'pressure',
         'datasets': [0], 'suptitle': 'Run A reference (pinned)'},
    ],
    title='Aggregations & references',
    ncols=2,
    height=360,
)

## 4. Multi-axis overlays and a live table

- **`plot_ymult`** overlays several channels on a *single* plot with independent
  y-axes &mdash; pressure, temp and flow at wildly different scales all read cleanly.
  `legend='right'` moves its legend out of the stack of axes.
- **`table`** is a panel too. Its `y` control picks the **columns**; giving `kwargs`
  an `x_in` list switches it into **interpolation mode**, reading each column at those
  `time` values (here 0, 20, 40, 59) with `sig_figs` rounding &mdash; a compact numeric
  readout beside the plots.

In [6]:
uc.line('time', 20)

In [7]:
uc.dashboard(
    panels=[
        {'method': 'plot_ymult', 'x': 'time', 'y': ['pressure', 'temp', 'flow'],
         'legend': 'right', 'suptitle': 'All channels, independent axes'},
        {'method': 'table',      'x': 'time', 'y': ['pressure', 'temp'],
         'kwargs': {'x_in': [0, 20, 40, 59], 'sig_figs': 3},
         'suptitle': 'Readouts at t = 0, 20, 40, 59'},
    ],
    title='Overlays & readouts',
    ncols=2,
    width=560,
    height=400,
)

## 5. Contour engineering maps

Contour panels run on the **`uc2`** compressor world. A contour needs a **`z`** column
(its own dropdown appears for contour panels), and `kwargs={'overlay_sets': [...]}`
draws other datasets on top of the field &mdash; the surge line (set 1) and the measured
operating points (set 2), each keeping its own style. Both panels pin `datasets=[0]`
(the surface) and overlay the same two sets, so they read as one map from two angles.

In [8]:
uc2.dashboard(
    panels=[
        {'method': 'contour', 'x': 'FLOW', 'y': 'PR', 'z': 'EFF',
         'suptitle': 'Efficiency map + surge + ops',
         'datasets': [0], 'kwargs': {'overlay_sets': [1, 2]}},
        {'method': 'contour', 'x': 'FLOW', 'y': 'PR', 'z': 'MARGIN',
         'suptitle': 'Surge margin',
         'datasets': [0], 'kwargs': {'overlay_sets': [1, 2]}},
    ],
    title='Compressor map',
    ncols=2,
    width=560,
    height=440,
)

## 6. The everything board

Everything from levels 1&ndash;4 on one 3&times;3 grid over all three runs: overlays, stacked
totals, box spreads, a distribution, a pinned single-run trace, and an interpolation
table. Setting `uc.darkmode = True` before the call **seeds the theme switch to dark**
(flip it live in the header). `ncols=3` lays the seven panels out in three rows.

This is the shape of a real operations board: one header driving a consistent slice,
each panel a different lens on it, a couple pinned as fixed references.

In [14]:
# uc.table(['flow', 'pressure'])

In [20]:
uc.linestyle('all', "-")

In [18]:
for s in uc.sets:
    uc.reg_order(s, 2)

In [22]:
uc.delta(0, [1,2], align_on='time', delta_parms=['rpm', 'pressure', 'temp', 'flow'])

Loaded Set 3: Delta 0-1
Loaded Set 4: Delta 0-2


[<unichart.Dataset at 0x77232c3f0b90>, <unichart.Dataset at 0x77232c3f1a90>]

In [29]:
panels=[
    {'method': 'plot_ymult', 
        'x': 'time', 
        'y': ['pressure', 'temp', 'flow'],
        'legend': 'right', 
        'suptitle': 'Channels (independent axes)'
    },
    {'method': 'bar',        
        'x': 'phase', 
        'y': ['flow', 'pressure'],
        'kwargs': {'barmode': 'stack', 
                    'agg': 'sum'}, 
                    'suptitle': 'Stacked totals'
    },
    {'method': 'box',        
        'x': 'phase', 
        'y': 'vibration', 
        'suptitle': 'Vibration spread'
    },
    {'method': 'histogram',  
        'x': 'temp',  'y': 'temp',
        'kwargs': {'nbins': 25}, 
        'suptitle': 'Temp distribution'
    },
    {'method': 'plot',       
        'x': 'time',  
        'y': 'rpm', 
        'suptitle': 'RPM'},
    {'method': 'plot',       
        'x': 'time',  
        'y': 'pressure',
        'datasets': [0], 
        'suptitle': 'Run A pressure (pinned)'
    },
    {'method': 'table',      
        'x': 'time',  
        'y': ['pressure', 'flow'],
        'kwargs': 
            {'x_in': [10, 30, 50], 
                'sig_figs': 4}, 
        'suptitle': 'Readouts'
    },
    {'method': 'table',      
        'x': 'time',  
        'y': ['DL_pressure', 'DL_rpm'],
        'datasets': [3,4], 
        'kwargs': 
            {'x_in': [10, 30, 50], 
                'sig_figs': 4}, 
        'suptitle': 'Deltas'
    },
]

In [34]:
uc.display_parms = ['pressure', 'temp', 'flow']

In [35]:
uc.darkmode = True   # seed the header's theme switch to dark
uc.omit([3,4])
uc.dashboard(
    panels=[
        {'method': 'plot_ymult', 
            'x': 'time', 
            'y': ['pressure', 'temp', 'flow'],
            'legend': 'right', 
            'suptitle': 'Channels (independent axes)'
        },
        {'method': 'bar',        
            'x': 'phase', 
            'y': ['flow', 'pressure'],
            'kwargs': {'barmode': 'stack', 
                        'agg': 'sum'}, 
                        'suptitle': 'Stacked totals'
        },
        {'method': 'box',        
            'x': 'phase', 
            'y': 'vibration', 
            'suptitle': 'Vibration spread'
        },
        {'method': 'histogram',  
            'x': 'temp',  'y': 'temp',
            'kwargs': {'nbins': 25}, 
            'suptitle': 'Temp distribution'
        },
        {'method': 'plot',       
            'x': 'time',  
            'y': 'rpm', 
            'suptitle': 'RPM'},
        {'method': 'plot',       
            'x': 'time',  
            'y': 'pressure',
            'datasets': [0], 
            'suptitle': 'Run A pressure (pinned)'
        },
        {'method': 'table',      
            'x': 'time',  
            'y': ['pressure', 'flow'],
            'kwargs': 
                {'x_in': [10, 30, 50], 
                 'sig_figs': 4}, 
            'suptitle': 'Readouts'
        },
        {'method': 'table',      
            'x': 'time',  
            'y': ['DL_pressure', 'DL_rpm'],
            'datasets': [3,4], 
            'kwargs': 
                {'x_in': [10, 30, 50], 
                 'sig_figs': 4}, 
            'suptitle': 'Deltas'
        },
    ],
    title='Rig operations board',
    ncols=3+1,
    width=460*1.2,
    height=340*1.2,
    controls=False,
)

In [37]:
path = uc.dashboard_to_html(panels, 'dashboard_progression_demo.html',
                            title='Rig overview', embed_js=True,
    # title='Rig operations board',
    ncols=3+1,
    width=460*1.2,
    height=340*1.2,)

In [10]:
uc.darkmode = False   # tidy up so later plain-notebook cells render light

## Notes

- **Run target.** `jupyter_mode='inline'` (default) renders in the cell. Use
  `jupyter_mode='external'` (or `'tab'`) to open a board in a browser instead.
- **One schema per board.** Non-pinned panels all plot the header's selected datasets,
  so a single board expects every set to share a schema. Genuinely different-shaped
  data (the contour field) belongs on its own notebook &mdash; or pin each panel.
- **`kwargs` are method-scoped.** A kwarg is passed only to methods that accept it, so
  it stays valid when you flip a panel's plot type live.
- **Sizing.** `width`/`height` are explicit px per panel &mdash; explicit sizes are what
  make panels paint reliably in the inline Jupyter iframe.